# Lab 03: acquire small scientific inputs and explain failures

Automated Workflows in Satellite Image Processing · v1.0 · 15 October 2026

Lab 02 discovered catalogue items and stored one preview. This lab acquires the red
and near-infrared bands needed for NDVI, but stores only a small area of interest.
You design the new database tables completely and then use the resulting archive
to investigate failures.

Use this notebook for predictions, SQL inspection and explanations. Edit
`student_schema.sql` in your editor. Run acquisition and failure cases in a terminal.
The supplied Python is working code, not a new function-repair exercise.

Deliverables: your full schema SQL, your diagnostic SQL, this notebook with evidence
and explanations, and a small final JSON report. Core work: approximately two hours.


## 1. Setup and continuity

Read README.md for installation. From a terminal in this folder, install Rasterio,
NumPy, Matplotlib and Jupyter into your existing environment if needed:

```bash
conda install -c conda-forge "rasterio>=1.4,<2" numpy matplotlib jupyterlab ipykernel
```

We preserve Lab 02's streams, scenes, downloads, runs and stream_scenes tables. You
can copy your database with --from-lab2 on the first init; README.md gives the command.
The default path creates a separate Lab 03 archive and uses captured actual Lab 01
metadata. The original Lab 02 database is never edited by the copy operation.

The fixed example is from June 2025. It is chosen for reproducibility, not presented
as the newest acquisition. Live acquire commands always read remote COG data;
packaged real subsets are used only by offline checks.


In [10]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

PROJECT = Path.cwd().resolve()
assert (PROJECT / 'lab03_cli.py').is_file(), 'Set PROJECT to the extracted lab folder'

def run_checks():
    result = subprocess.run([sys.executable, str(PROJECT / 'check_lab03.py')], cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('Exit code:', result.returncode)
    return result.returncode

print('Project:', PROJECT)


Project: /Users/jklimek/Desktop/AGH/AUTOMATED_WORKFLOWS/Labs/Lab03


## 2. Design five complete tables (45–60 minutes)

Read SCHEMA_CONTRACT.md. Write every CREATE TABLE statement yourself in
student_schema.sql. The file initially contains only comments. Required column
names let the supplied service use your schema, but there are no partial table
statements to complete.

Before writing SQL, sketch these relationships and explain the expected cardinality:

| Parent | Child | Question |
|---|---|---|
| scenes | assets | Can one scene have more than one band or metadata revision? |
| assets | subset_requests | Can the same band be stored for two AOIs? |
| subset_requests | download_attempts | Can a request fail and later succeed? |
| runs | download_attempts | Can one invocation attempt more than one band? |
| download_attempts | artifacts | Which attempt first registered these exact bytes? |
| artifacts | artifact_locations | Can a stored file disappear without erasing the artifact's identity? |

Also record each request's latest artifact and each attempt's result artifact.
Those artifacts must belong to the same request, not simply be any existing ID.
The contract explains the composite references needed for that condition.

Which facts belong to current state, and which must remain as history? Do not
make run_id or subset_id unique in the attempts table. Predict what that would
prevent during recovery.


In [11]:
run_checks()


PASS: five complete table definitions and column contract
PASS: primary keys and scene/asset/request/run foreign keys
PASS: AOI identity, uniqueness and allowed states
PASS: attempt outcomes, timestamps and byte counts
PASS: composite relationships prevent cross-AOI artifact references
PASS: real-file verification, missing/corrupt diagnosis and preserved success history
PASS: real B04/B08 grids, AOI masking, scale/offset and pixel cap
PASS: server ignoring Range is refused without reading its body
All checks used actual captured EO data; no live network requests were made.

attempt=attempt-bff667a4-4b5b-42a9-89c4-05508f60514e code=FILE_MISSING Recorded file is absent: real_red_aoi.tif
attempt=attempt-40b4311c-6504-423d-a050-75a6a69bffd4 code=FILE_CORRUPT Size/hash mismatch for real_red_aoi.tif

Exit code: 0


0

*Write your schema explanation here. Identify two uniqueness constraints and two foreign key constraints, and give a concrete bad record each would reject. Explain why an asset+band identity alone is insufficient when the AOI changes.*

**Two uniqueness constraints**
- CONSTRAINT u_assets UNIQUE(collection, item_id, asset_key, metadata_sha256) - this constraint would reject any already existing asset with new asset_id
- CONSTRAINT u_subset_requests UNIQUE(asset_id, aoi_key, recipe_version) - rejects two subset_ids for the same asset, aoi and recipe.

**Two foreign key constraints**
- assets | FOREIGN KEY (collection, item_id) REFERENCES scenes(collection, item_id) - it rejects an asset which collection and item_id is not present in scenes
- subset_requests | FOREIGN KEY (latest_artifact_id, subset_id) REFERENCES artifacts(artifact_id, subset_id) - it rejects subset_request which latest_artifact_id and subset_id are not in artifacts

**Why an asset + band is not enough?**

Because then we would consider different cutouts the same.


## 3. Plan before storing scientific data (10 minutes)

Once checks pass, initialize the new archive and inspect the planned size:

```bash
python lab03_cli.py init --config config/krakow_aoi.json
```

```bash
python lab03_cli.py plan --config config/krakow_aoi.json
```

Open config/krakow_aoi.json. Distinguish discovery_bbox, which selects catalogue
candidates, from aoi_bbox, which limits the stored pixels. Both use west, south,
east, north in EPSG:4326. item_id pins one tile; the AOI must fit entirely inside it.

For each band, report source_shape, subset shape, pixel size, CRS, raw pixel bytes,
scale and offset. The supplied default should produce 227 rows by 219 columns at
10 m resolution, much smaller than the 10,980 by 10,980 source tile.

Use the alternative too_large_aoi.json configuration to investigate a budget failure:

```bash
python lab03_cli.py plan --config config/too_large_aoi.json
```

That command is expected to fail before remote raster data is read. Explain why a
local disk limit and an HTTP transfer limit are different controls. Reading a COG
window fetches whole source blocks; network bytes may exceed stored file bytes.


## 4. Acquire and repeat (10 minutes plus network time)

```bash
python lab03_cli.py acquire --config config/krakow_aoi.json
```

```bash
python lab03_cli.py acquire --config config/krakow_aoi.json
```

```bash
python lab03_cli.py report --config config/krakow_aoi.json
```

The first run requests two real bands via byte ranges and stores two masked AOI
GeoTIFFs. The second checks their size, SHA-256 and raster metadata and records
reused attempts. Explain why reused is not another transfer and why it is still
worth recording.

*The second run is not a transfer but rather a check if the local tiff matches the archived artifact.*

Open state/catalogue.sqlite in DB Browser. Inspect the five original tables and
your five new tables. Existing preview downloads remain in downloads; scientific
cutout attempts are in download_attempts and link to subset_requests.


### One worked SQL join

Run this in Execute SQL. It follows an attempt to its run, AOI request and source
band. Use it as a starting point for your own diagnostic queries.

```sql
SELECT r.run_id, r.command, a.item_id, a.asset_key,
       s.aoi_key, d.status, d.stage, d.network_bytes,
       d.http_status, d.error_code, d.error_message
FROM download_attempts AS d
JOIN runs AS r ON r.run_id = d.run_id
JOIN subset_requests AS s ON s.subset_id = d.subset_id
JOIN assets AS a ON a.asset_id = s.asset_id
ORDER BY d.started_at, d.attempt_id;
```

Write your own query to show the saved paths, file hashes and latest location
states. Compare HTTP bytes with file sizes. Do not sum bytes after joining a
one-to-many location table unless you account for the duplicated attempt rows.

```sql
SELECT a.asset_key,
       d.status AS attempt_status,
       d.network_bytes AS http_bytes,
       art.size_bytes AS file_size_bytes,
       art.sha256 AS file_hash,
       loc.local_path,
       loc.status AS location_status
FROM download_attempts AS d
JOIN subset_requests AS s ON s.subset_id = d.subset_id
JOIN assets AS a ON a.asset_id = s.asset_id
LEFT JOIN artifacts AS art ON art.artifact_id = d.artifact_id
LEFT JOIN artifact_locations AS loc ON loc.artifact_id = art.artifact_id
ORDER BY d.started_at, d.attempt_id;
```


In [12]:
DB = PROJECT / 'state' / 'catalogue.sqlite'
if not DB.is_file():
    print('Write/check the schema and run init/acquire in the terminal first.')
else:
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        names = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if 'download_attempts' not in names:
            print('The five new tables have not been initialized yet.')
        else:
            conn.row_factory = sqlite3.Row
            rows = [dict(row) for row in conn.execute('SELECT run_id,status,stage,network_bytes,error_code FROM download_attempts ORDER BY rowid')]
            print(json.dumps(rows, indent=2))


[
  {
    "run_id": "lab03-7789319d-c978-4fda-995c-e9191427599a",
    "status": "failed",
    "stage": "read_window",
    "network_bytes": 1703937,
    "error_code": "RANGE_READ_FAILED"
  },
  {
    "run_id": "lab03-efa7ef85-ba68-4827-b3dd-933f1754ae58",
    "status": "succeeded",
    "stage": "complete",
    "network_bytes": 2097153,
    "error_code": null
  },
  {
    "run_id": "lab03-efa7ef85-ba68-4827-b3dd-933f1754ae58",
    "status": "succeeded",
    "stage": "complete",
    "network_bytes": 2097153,
    "error_code": null
  },
  {
    "run_id": "lab03-87031363-90d1-40e1-90c1-899dc2dd8de3",
    "status": "reused",
    "stage": "cache_verify",
    "network_bytes": 0,
    "error_code": null
  },
  {
    "run_id": "lab03-87031363-90d1-40e1-90c1-899dc2dd8de3",
    "status": "reused",
    "stage": "cache_verify",
    "network_bytes": 0,
    "error_code": null
  }
]


## 5. Failure case A: one band succeeded, then the connection failed (20 minutes)

Predict the database and file states before running this:

```bash
python lab03_cli.py case --config config/krakow_aoi.json --name network
```

The command creates cases/network/config.json and its own database/data directory,
then deliberately interrupts the nir reader after actual range responses. A
failure exit code is expected. It does not damage the main archive. The connection
failure is controlled; the scene records and imagery are real.

Open cases/network/state/catalogue.sqlite. Before any recovery:

1. Write SQL to identify the failed run, failed band, failure stage and bytes received.
2. Show which band already has an artifact and which has none.
3. Explain why a failed run does not imply that every operation in it failed. How could an INNER JOIN to artifacts hide the failed attempt? *Inner joins drops rows with no artifact_ids, thus it drops rows that were not successful. Left join keeps match even when artifact id is missing.*
4. Explain why advancing a discovery watermark would not prove a band was stored. *Watermark is stored with respect to search over the catalogue. It does not necessarily mean the asset was stored properly.*

```sql
SELECT r.run_id, a.asset_key, d.stage, d.network_bytes,
       d.error_code, d.error_message
FROM download_attempts d
JOIN runs r ON r.run_id = d.run_id
JOIN subset_requests s ON s.subset_id = d.subset_id
JOIN assets a ON a.asset_id = s.asset_id
WHERE d.status = 'failed';
```

```sql
SELECT a.asset_key, d.status, d.stage, d.artifact_id
FROM download_attempts d
JOIN subset_requests s ON s.subset_id = d.subset_id
JOIN assets a ON a.asset_id = s.asset_id
WHERE d.run_id = 'lab03-419164a6-e0b1-413c-9cbc-129c1c7aef35'
ORDER BY d.started_at;
```

Record evidence, then recover:

```bash
python lab03_cli.py acquire --config cases/network/config.json
```

Show that the red file was verified/reused, the nir operation was retried, and the
original failure remains in the attempts table. Do not overwrite the failed row
to make the history look successful. Finish with a small before/after table of
run, band, attempt status, stage and artifact identity.

```sql
SELECT r.run_id, r.status AS run_status, a.asset_key AS band,
       d.status AS attempt_status, d.stage, d.artifact_id, d.error_code
FROM download_attempts d
JOIN runs r ON r.run_id = d.run_id
JOIN subset_requests s ON s.subset_id = d.subset_id
JOIN assets a ON a.asset_id = s.asset_id
ORDER BY d.started_at, d.attempt_id;
```

![L6 EVIDENCE](./L6_RUNS.png)


## 6. Failure case B: recorded success, missing file (20 minutes)

Predict whether a foreign key can detect a missing disk file. Then run:

```bash
python lab03_cli.py case --config config/krakow_aoi.json --name missing
```

This creates an independent case, acquires real subsets, deletes only that case's
red file, and verifies the archive. A failure exit code is expected. Inspect
cases/missing/state/catalogue.sqlite before recovery.

Write diagnostic SQL and explain:

- Which historical attempt succeeded? Which later operation detected the loss? 

  *Acquire succeeded and verify detected the loss.*
- Which table describes the file's current availability, and which describes the
  bytes that previously existed? 

  *artifacts_locations - current state on the disk, artifacts - bytes*
- Why should the original successful attempt remain successful in the history? 

  *I don't think file deletion should be related to history entry deletion. Keeping it in database ensures integrity because that acquisition was done and should be persisted.*
- What can the database prove, and what required reading the filesystem? 

  *DB proves recorded path, hash, and download attempt. But to check if file is actually on disk can be only proven by programatic check, not some DB entries. DB won't be updated when the file is being deleted.*

```sql
SELECT r.run_id, r.command, r.status AS run_status,
       a.asset_key AS band, d.status AS attempt_status, d.stage,
       d.error_code, d.artifact_id
FROM download_attempts d
JOIN runs r ON r.run_id = d.run_id
JOIN subset_requests s ON s.subset_id = d.subset_id
JOIN assets a ON a.asset_id = s.asset_id
ORDER BY d.started_at;
```

```sql
SELECT a.asset_key, art.artifact_id, art.sha256, art.size_bytes,
       loc.status AS location_status, loc.local_path, loc.checked_at
FROM artifacts art
JOIN subset_requests s ON s.subset_id = art.subset_id
JOIN assets a ON a.asset_id = s.asset_id
JOIN artifact_locations loc ON loc.artifact_id = art.artifact_id
ORDER BY a.asset_key;
```

Recover with the same configuration:

```bash
python lab03_cli.py acquire --config cases/missing/config.json
```

Confirm red was restored and nir was reused. If the restored GeoTIFF has the same
hash, explain why it may retain the same artifact identity while having a new
successful transfer attempt.

![L7_1](./L7_1.png)
![L7_2](./L7_2.png)

Optional: --name corrupt truncates a real case file instead of deleting it.
Compare FILE_CORRUPT with FILE_MISSING. --name interrupted stops after writing a
valid temporary cutout but before finalization; explain why no successful artifact
should be registered for that attempt. Generate each case once; later use its
existing configuration to inspect or recover it.


## 7. A new AOI is a different request (10 minutes)

Compare the plan for a smaller AOI of the same scene:

```bash
python lab03_cli.py plan --config config/alternate_aoi.json
```

Optional small acquisition:

```bash
python lab03_cli.py acquire --config config/alternate_aoi.json
```

```sql
SELECT a.asset_id, count(s.subset_id) FROM assets a JOIN subset_requests s ON a.asset_id = s.asset_id GROUP BY a.asset_id 
```

Write SQL showing that the same source asset can have two subset requests and
that each request points to the correct output. Why would UNIQUE(asset_id) on
subset_requests break this? Why would two independent foreign keys fail to prove
that an artifact belongs to the right request?

*UNIQUE(asset_id) would allow only one request for one asset_id. As we know it is not the case because the subset_requests vary not only on asset_id but also on aoi_key and recipe.*


## 8. Optional inspection: prepare for NDVI processing

B04/red and B08/nir are the two 10 m inputs for this exercise. Inspect their
reflectance, not unsigned integer arithmetic. Rasterio does not automatically
apply the stored scale and offset when read() returns an array. The offset can
change NDVI; assuming all calibration factors cancel is unsafe.

The cell below reads your main archive only. It does not download anything or
register a derived product. NDVI is shown for inspection; cloud masking, formal
QC and product registration belong to later work.


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

paths = {}
if DB.is_file():
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        names = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if 'artifact_locations' in names:
            for band, path in conn.execute("""SELECT a.asset_key,l.local_path FROM subset_requests s
                JOIN assets a USING(asset_id) JOIN artifact_locations l ON l.artifact_id=s.latest_artifact_id
                WHERE l.status='available' ORDER BY s.updated_at"""):
                paths[band] = Path(path)
if set(paths) != {'red','nir'} or not all(p.is_file() for p in paths.values()):
    print('Acquire both main-archive AOI subsets first; no download is performed by this cell.')
else:
    arrays = {}
    grids = {}
    for band, path in paths.items():
        with rasterio.open(path) as src:
            grids[band] = (src.crs, src.transform, src.shape)
            arrays[band] = src.read(1, masked=True).astype('float32') * src.scales[0] + src.offsets[0]
            print(band, src.shape, src.crs, 'scale:', src.scales[0], 'offset:', src.offsets[0])
    assert grids['red'] == grids['nir'], 'Choose the same AOI and aligned bands'
    red, nir = arrays['red'], arrays['nir']
    denominator = nir + red
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = np.ma.masked_where(denominator <= 0, (nir-red) / denominator)
    fig, axes = plt.subplots(1,3,figsize=(12,4))
    for ax, values, title in zip(axes,[red,nir,ndvi],['Red reflectance','NIR reflectance','NDVI: inspection only']):
        im = ax.imshow(values, cmap='RdYlGn' if title.startswith('NDVI') else 'gray')
        ax.set_title(title); ax.set_xlabel('Column'); ax.set_ylabel('Row')
        fig.colorbar(im,ax=ax,shrink=0.65)
    plt.tight_layout()
    plt.show()


## Submission and interpretation

Submit student_schema.sql, your investigation_queries.sql, this notebook with
predictions/SQL results/explanations, and a small report:

```bash
python lab03_cli.py report --config config/krakow_aoi.json > lab03_report.json
```

For each required failure, state: symptom, query, observed rows, diagnosis, recovery,
and post-recovery evidence. A screenshot alone is not a diagnosis. Explain which
claims are protected by database constraints and which require checking bytes,
filesystem state or raster metadata.

Keep state/, data/, and cases/ out of Git. Successful offline checks do not claim
that a live request succeeded. If the provider/network is unavailable, retain that
failure evidence and complete the live acquisition later; no synthetic image is
substituted.

Sources: [Rasterio windowed reading/writing](https://rasterio.readthedocs.io/en/latest/topics/windowed-rw.html)
and [Earth Search metadata](https://github.com/Element84/earth-search/blob/main/README.md).
